In [1]:
!pip install requests beautifulsoup4 pandas

In [2]:
pip install --upgrade pip

Note: you may need to restart the kernel to use updated packages.


In [4]:
!pip install selenium webdriver-manager

In [ ]:
import requests
import pandas as pd
import time
import os
from datetime import datetime, timedelta

output_path = os.path.expanduser("~/Desktop/vietnam_articles.csv")

headers = {"User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 Chrome/120.0.0.0 Safari/537.36"}

articles = []
seen_urls = set()

# ── APPROACH: VnExpress sitemap XML ──
# VnExpress publishes a sitemap with ALL article URLs
# This is the most reliable way to get historical articles

print("📡 Fetching VnExpress sitemap index...")

# VnExpress sitemap index - lists all monthly sitemaps
sitemap_index = "https://vnexpress.net/sitemap/news.rss"

# Monthly sitemaps follow this pattern
# https://vnexpress.net/sitemap/kinh-doanh-2024-01.xml

sitemap_urls = []
for year in range(2018, 2026):
    for month in range(1, 13):
        for section in ["kinh-doanh", "kinh-doanh/chung-khoan"]:
            sitemap_urls.append(
                f"https://vnexpress.net/sitemap/{section}-{year}-{month:02d}.xml"
            )

print(f"Generated {len(sitemap_urls)} sitemap URLs")

# Test first one
test = requests.get(sitemap_urls[0], headers=headers, timeout=15)
print(f"Test status: {test.status_code}")
print(f"Preview: {test.text[:300]}")

📡 Fetching VnExpress sitemap index...
Generated 192 sitemap URLs
Test status: 406
Preview: <h1>406 Not Acceptable</h1>


In [ ]:
import requests
import pandas as pd
import xml.etree.ElementTree as ET
import re
import time
import os

output_path = os.path.expanduser("~/Desktop/vietnam_articles.csv")
articles = []
seen_urls = set()

headers = {"User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 Chrome/120.0.0.0 Safari/537.36"}

# Google News RSS - no blocking, returns real articles
# These search Vietnamese financial keywords via Google News
google_rss_queries = [
    "chứng+khoán+Việt+Nam",
    "VN-Index+thị+trường",
    "cổ+phiếu+HOSE",
    "thị+trường+chứng+khoán",
    "nhà+đầu+tư+chứng+khoán",
    "VnIndex+tăng+giảm",
    "chứng+khoán+HNX",
    "cổ+phiếu+tăng+giảm",
    "thị+trường+tài+chính+Việt+Nam",
    "chứng+khoán+thanh+khoản",
]

for query in google_rss_queries:
    url = f"https://news.google.com/rss/search?q={query}&hl=vi&gl=VN&ceid=VN:vi"
    print(f"\n🔍 {query.replace('+', ' ')}")
    
    try:
        resp = requests.get(url, headers=headers, timeout=15)
        print(f"  Status: {resp.status_code}")
        
        root = ET.fromstring(resp.content)
        items = root.findall(".//item")
        print(f"  Items: {len(items)}")
        
        for item in items:
            title = item.findtext("title", "").strip()
            link  = item.findtext("link",  "").strip()
            date  = item.findtext("pubDate", "").strip()
            desc  = re.sub(r'<[^>]+>', '', item.findtext("description", "")).strip()
            src   = item.findtext("source", "").strip()
            
            if title and link and link not in seen_urls:
                articles.append({
                    "title":       title,
                    "url":         link,
                    "date":        date,
                    "description": desc,
                    "source":      src or "Google News",
                    "keyword":     query.replace("+", " ")
                })
                seen_urls.add(link)
                print(f"  ✅ {title[:70]}")
        
        time.sleep(2)
        
    except Exception as e:
        print(f"  ❌ Error: {e}")

# Save
df = pd.DataFrame(articles).drop_duplicates(subset=["url"])
df.to_csv(output_path, index=False, encoding="utf-8-sig")
print(f"\n✅ {len(df)} articles saved to {output_path}")
print(df[["title","date","source"]].head(10).to_string())


🔍 chứng khoán Việt Nam
  Status: 200
  Items: 102
  ✅ Chứng khoán Việt Nam đứng trước cơ hội “tái định giá” sau nâng hạng - 
  ✅ Blog chứng khoán: Tiền lớn chưa “động” - VnEconomy
  ✅ Cổ phiếu cần quan tâm ngày 22/9 - Tin nhanh chứng khoán
  ✅ Fed trở lại với chu kỳ tăng lãi suất: Chứng khoán Việt Nam đứng trước 
  ✅ Fed tăng lãi suất, dòng tiền vào chứng khoán khó cải thiện trong ngắn 
  ✅ Fed tăng lãi suất: Chứng khoán Việt Nam tìm lực đỡ từ đâu? - Tin nhanh
  ✅ VFA 2026 - Lễ trao Giải thưởng Quỹ đầu tư chứng khoán Việt Nam sẽ diễn
  ✅ Morgan Stanley: Thanh khoản chứng khoán Việt Nam có thể tăng 5 lần sau
  ✅ Thị trường chứng khoán Việt Nam chuyển từ tăng quy mô sang nâng chất l
  ✅ Điều gì sẽ xảy ra với chứng khoán Việt Nam sau ngày 21/9? - Znews
  ✅ Chứng khoán tuần mới: Sau nâng hạng, VN-Index sẽ ra sao? - Tuổi Trẻ
  ✅ Chứng khoán giảm điểm ngày đầu nâng hạng, chuyên gia kiến nghị quan tr
  ✅ [Infographic] Chứng khoán ngày 21/9: VN-Index giảm 15,99 điểm, khối lư
  ✅ Chứng khoán V

In [ ]:
import requests
from bs4 import BeautifulSoup

headers = {
    "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    "Accept-Language": "vi-VN,vi;q=0.9,en-US;q=0.8",
    "Referer": "https://vnexpress.net",
}

url = "https://timkiem.vnexpress.net/?q=VN-Index&cate_code=1004&from_date=1514764800&to_date=1767225600&page=1"

resp = requests.get(url, headers=headers, timeout=15)
resp.encoding = "utf-8"
soup = BeautifulSoup(resp.text, "html.parser")

print(f"Status: {resp.status_code}")
print(f"Total <article> tags: {len(soup.select('article'))}")
print(f"Total <a> tags: {len(soup.find_all('a'))}")

# Print first 20 links with titles
for a in soup.find_all("a", href=True)[:30]:
    title = a.get_text(strip=True)
    href  = a.get("href", "")
    if len(title) > 20 and "vnexpress" in href:
        print(f"  {href[:80]} | {title[:60]}")

Status: 200
Total <article> tags: 26
Total <a> tags: 164


In [ ]:
import requests
import pandas as pd
import xml.etree.ElementTree as ET
from bs4 import BeautifulSoup
import re
import time
import os

output_path = os.path.expanduser("~/Desktop/vietnam_articles.csv")

# Load existing if any
try:
    df_existing = pd.read_csv(output_path)
    articles = df_existing.to_dict("records")
    seen_urls = set(df_existing["url"].tolist())
    print(f"✅ Loaded {len(articles)} existing articles")
except:
    articles = []
    seen_urls = set()

headers = {
    "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    "Accept-Language": "vi-VN,vi;q=0.9,en-US;q=0.8",
    "Referer": "https://vnexpress.net",
}

def save(label=""):
    df = pd.DataFrame(articles).drop_duplicates(subset=["url"])
    df.to_csv(output_path, index=False, encoding="utf-8-sig")
    print(f"  💾 {label} Saved: {len(df)} articles")
    return df

# ══════════════════════════════════════════
# SOURCE 1: Google News RSS (102 per query)
# ══════════════════════════════════════════
print("\n" + "="*55)
print("SOURCE 1: Google News RSS")
print("="*55)

google_queries = [
    "chứng+khoán+Việt+Nam",
    "VN-Index+thị+trường",
    "cổ+phiếu+HOSE+HNX",
    "thị+trường+chứng+khoán+Việt",
    "nhà+đầu+tư+chứng+khoán",
    "VnIndex+tăng+giảm+điểm",
    "chứng+khoán+HNX+thanh+khoản",
    "cổ+phiếu+tăng+giảm+mạnh",
    "thị+trường+tài+chính+Việt+Nam",
    "chứng+khoán+biến+động",
    "cổ+phiếu+ngân+hàng+chứng+khoán",
    "VN30+index+chứng+khoán",
    "chứng+khoán+phái+sinh",
    "thị+trường+upcom+chứng+khoán",
    "cổ+phiếu+bất+động+sản",
    "chứng+khoán+tuần+tháng",
    "nhịp+đập+thị+trường+chứng+khoán",
    "tâm+lý+nhà+đầu+tư+chứng+khoán",
    "dòng+tiền+chứng+khoán",
    "chứng+khoán+ngoại+khối+bán+mua",
    "lạm+phát+lãi+suất+chứng+khoán",
    "kinh+tế+vĩ+mô+chứng+khoán",
    "chứng+khoán+COVID+dịch+bệnh",
    "chứng+khoán+2018+2019+2020",
    "chứng+khoán+2021+2022+2023",
    "chứng+khoán+2024+2025",
    "Vietnam+stock+market+VN-Index",
    "Vietnam+stock+exchange+HOSE",
    "Vietnam+securities+investor",
    "Vietnam+stock+market+volatility",
]

for i, query in enumerate(google_queries):
    url = f"https://news.google.com/rss/search?q={query}&hl=vi&gl=VN&ceid=VN:vi"
    try:
        resp = requests.get(url, headers=headers, timeout=15)
        root = ET.fromstring(resp.content)
        items = root.findall(".//item")
        
        added = 0
        for item in items:
            title = item.findtext("title", "").strip()
            link  = item.findtext("link",  "").strip()
            date  = item.findtext("pubDate", "").strip()
            src   = item.findtext("source", "").strip()
            desc  = re.sub(r'<[^>]+>', '', item.findtext("description","")).strip()
            
            if title and link and link not in seen_urls:
                articles.append({
                    "title": title, "url": link, "date": date,
                    "description": desc, "source": src,
                    "keyword": query.replace("+", " ")
                })
                seen_urls.add(link)
                added += 1
        
        print(f"  [{i+1:2d}/{len(google_queries)}] {query[:40]:<40} | +{added:3d} | Total: {len(articles):6d}")
        time.sleep(1.5)
        
    except Exception as e:
        print(f"  ❌ {query[:40]}: {e}")

save("After Google News RSS")

# ══════════════════════════════════════════
# SOURCE 2: VnExpress Search (26 per page)
# ══════════════════════════════════════════
print("\n" + "="*55)
print("SOURCE 2: VnExpress Search Engine")
print("="*55)

vne_keywords = [
    "VN-Index", "chứng khoán", "cổ phiếu", "HOSE",
    "thị trường chứng khoán", "nhà đầu tư", "thanh khoản",
    "tăng điểm", "giảm điểm", "HNX", "VN30",
    "phái sinh", "UPCOM", "cổ phần",
    "ngân hàng chứng khoán", "bất động sản",
    "lãi suất", "tỷ giá", "lạm phát",
    "dòng tiền", "ngoại khối", "margin",
]

# from_date = Jan 1 2018, to_date = Dec 31 2025
FROM_DATE = 1514764800
TO_DATE   = 1767225600

for keyword in vne_keywords:
    print(f"\n  🔍 {keyword}")
    
    for page in range(1, 100):
        url = (
            f"https://timkiem.vnexpress.net/?q={requests.utils.quote(keyword)}"
            f"&cate_code=1004&from_date={FROM_DATE}&to_date={TO_DATE}&page={page}"
        )
        try:
            resp = requests.get(url, headers=headers, timeout=15)
            resp.encoding = "utf-8"
            soup = BeautifulSoup(resp.text, "html.parser")
            
            found = []
            for art in soup.select("article.item-news, div.item-news-common"):
                t    = art.select_one("h3.title-news a, h2.title-news a, a.title-news")
                d    = art.select_one("span.time-count, span.date, time, span.meta-news")
                desc = art.select_one("p.description")
                if not t:
                    continue
                link  = t.get("href", "")
                title = t.get_text(strip=True)
                if link and link not in seen_urls and len(title) > 15:
                    found.append({
                        "title": title, "url": link,
                        "date":  d.get_text(strip=True) if d else "",
                        "description": desc.get_text(strip=True) if desc else "",
                        "source": "VnExpress",
                        "keyword": keyword
                    })
                    seen_urls.add(link)
            
            articles.extend(found)
            print(f"    Page {page:3d} | +{len(found):3d} | Total: {len(articles):6d}")
            
            if len(found) == 0:
                break
            if page % 20 == 0:
                save(f"Checkpoint {keyword}")
            time.sleep(2)
            
        except Exception as e:
            print(f"    ❌ Page {page}: {e}")
            time.sleep(5)
            break
    
    save(f"After '{keyword}'")

# ══════════════════════════════════════════
# FINAL
# ══════════════════════════════════════════
df = save("FINAL")
print(f"\n{'='*55}")
print(f"✅ COMPLETE: {len(df)} articles")
print(f"📁 {output_path}")
print(f"\nBy source:\n{df['source'].value_counts().to_string()}")
print(f"\nSample:\n{df[['title','date','source']].head(10).to_string()}")

✅ Loaded 736 existing articles

SOURCE 1: Google News RSS
  [ 1/30] chứng+khoán+Việt+Nam                     | +  0 | Total:    736
  [ 2/30] VN-Index+thị+trường                      | +  0 | Total:    736
  [ 3/30] cổ+phiếu+HOSE+HNX                        | + 84 | Total:    820
  [ 4/30] thị+trường+chứng+khoán+Việt              | +  3 | Total:    823
  [ 5/30] nhà+đầu+tư+chứng+khoán                   | +  1 | Total:    824
  [ 6/30] VnIndex+tăng+giảm+điểm                   | + 47 | Total:    871
  [ 7/30] chứng+khoán+HNX+thanh+khoản              | + 73 | Total:    944
  [ 8/30] cổ+phiếu+tăng+giảm+mạnh                  | + 30 | Total:    974
  [ 9/30] thị+trường+tài+chính+Việt+Nam            | +  3 | Total:    977
  [10/30] chứng+khoán+biến+động                    | + 67 | Total:   1044
  [11/30] cổ+phiếu+ngân+hàng+chứng+khoán           | + 61 | Total:   1105
  [12/30] VN30+index+chứng+khoán                   | + 84 | Total:   1189
  [13/30] chứng+khoán+phái+sinh                    | +

In [7]:
import requests
import pandas as pd
import xml.etree.ElementTree as ET
from bs4 import BeautifulSoup
import re, time, os

output_path = os.path.expanduser("~/Desktop/vietnam_articles.csv")

# Load existing 10,962
df_existing = pd.read_csv(output_path)
articles = df_existing.to_dict("records")
seen_urls = set(df_existing["url"].tolist())
print(f"✅ Resuming from {len(articles)} articles")

headers = {
    "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 Chrome/120.0.0.0 Safari/537.36",
    "Accept-Language": "vi-VN,vi;q=0.9,en-US;q=0.8",
    "Referer": "https://vnexpress.net",
}

def save(label=""):
    df = pd.DataFrame(articles).drop_duplicates(subset=["url"])
    df.to_csv(output_path, index=False, encoding="utf-8-sig")
    print(f"  💾 {label}: {len(df)} articles")
    return df

# ══════════════════════════════════════════
# ROUND 2: More Google News queries
# ══════════════════════════════════════════
print("\n" + "="*55)
print("ROUND 2: Extended Google News RSS queries")
print("="*55)

new_google_queries = [
    # Year-specific queries to get historical articles
    "chứng+khoán+2018",
    "chứng+khoán+2019",
    "chứng+khoán+2020",
    "chứng+khoán+2021",
    "chứng+khoán+2022",
    "chứng+khoán+2023",
    "chứng+khoán+2024",
    "chứng+khoán+2025",
    "VN-Index+2018",
    "VN-Index+2019",
    "VN-Index+2020",
    "VN-Index+2021",
    "VN-Index+2022",
    "VN-Index+2023",
    "VN-Index+2024",
    # Sector-specific
    "cổ+phiếu+ngân+hàng+Việt+Nam",
    "cổ+phiếu+bất+động+sản+Việt+Nam",
    "cổ+phiếu+thép+Việt+Nam",
    "cổ+phiếu+dầu+khí+Việt+Nam",
    "cổ+phiếu+công+nghệ+Việt+Nam",
    "cổ+phiếu+tiêu+dùng+Việt+Nam",
    # Market events
    "chứng+khoán+COVID+2020+2021",
    "chứng+khoán+giảm+sàn+2022",
    "chứng+khoán+nâng+hạng+2024+2025",
    "chứng+khoán+MSCI+FTSE+Việt+Nam",
    "FED+lãi+suất+chứng+khoán+Việt",
    "tỷ+giá+USD+chứng+khoán+Việt+Nam",
    "room+ngoại+chứng+khoán+Việt+Nam",
    "chứng+khoán+margin+call+Việt",
    "thao+túng+chứng+khoán+Việt+Nam",
    # More sources
    "Vietnam+stock+VN-Index+2020+2021",
    "Vietnam+stock+market+crash+2022",
    "Vietnam+stock+upgrade+emerging+market",
    "Hanoi+Ho+Chi+Minh+stock+exchange",
    "Vietnam+securities+commission",
]

for i, query in enumerate(new_google_queries):
    url = f"https://news.google.com/rss/search?q={query}&hl=vi&gl=VN&ceid=VN:vi"
    try:
        resp = requests.get(url, headers=headers, timeout=15)
        root = ET.fromstring(resp.content)
        items = root.findall(".//item")
        
        added = 0
        for item in items:
            title = item.findtext("title", "").strip()
            link  = item.findtext("link", "").strip()
            date  = item.findtext("pubDate", "").strip()
            src   = item.findtext("source", "").strip()
            desc  = re.sub(r'<[^>]+>', '', item.findtext("description","")).strip()
            
            if title and link and link not in seen_urls:
                articles.append({
                    "title": title, "url": link, "date": date,
                    "description": desc, "source": src,
                    "keyword": query.replace("+", " ")
                })
                seen_urls.add(link)
                added += 1
        
        print(f"  [{i+1:2d}/{len(new_google_queries)}] {query[:45]:<45} | +{added:3d} | Total: {len(articles):6d}")
        time.sleep(1.5)
        
    except Exception as e:
        print(f"  ❌ {query[:45]}: {e}")

save("After Round 2 Google News")

# ══════════════════════════════════════════
# ROUND 3: VnExpress search - new keywords
# ══════════════════════════════════════════
print("\n" + "="*55)
print("ROUND 3: VnExpress - new keywords")
print("="*55)

FROM_DATE = 1514764800  # Jan 2018
TO_DATE   = 1767225600  # Dec 2025

new_vne_keywords = [
    "tăng trần", "giảm sàn", "room ngoại",
    "quỹ ETF", "phái sinh", "hợp đồng tương lai",
    "IPO", "niêm yết", "hủy niêm yết",
    "cổ tức", "chia tách cổ phiếu", "mua lại cổ phiếu",
    "thoái vốn", "M&A", "thâu tóm",
    "Vinhomes", "Vingroup", "Masan", "Techcombank",
    "Vietcombank", "BIDV", "Agribank", "VPBank",
    "Hòa Phát", "Sabeco", "Vinamilk", "FPT",
    "nâng hạng thị trường", "MSCI", "FTSE Russell",
    "SSC UBCKNN chứng khoán",
]

for keyword in new_vne_keywords:
    print(f"\n  🔍 {keyword}")
    for page in range(1, 50):
        url = (
            f"https://timkiem.vnexpress.net/?q={requests.utils.quote(keyword)}"
            f"&cate_code=1004&from_date={FROM_DATE}&to_date={TO_DATE}&page={page}"
        )
        try:
            resp = requests.get(url, headers=headers, timeout=15)
            resp.encoding = "utf-8"
            soup = BeautifulSoup(resp.text, "html.parser")
            
            found = []
            for art in soup.select("article.item-news, div.item-news-common"):
                t    = art.select_one("h3.title-news a, h2.title-news a, a.title-news")
                d    = art.select_one("span.time-count, span.date, time")
                desc = art.select_one("p.description")
                if not t:
                    continue
                link  = t.get("href", "")
                title = t.get_text(strip=True)
                if link and link not in seen_urls and len(title) > 15:
                    found.append({
                        "title": title, "url": link,
                        "date": d.get_text(strip=True) if d else "",
                        "description": desc.get_text(strip=True) if desc else "",
                        "source": "VnExpress", "keyword": keyword
                    })
                    seen_urls.add(link)
            
            articles.extend(found)
            print(f"    Page {page:3d} | +{len(found):3d} | Total: {len(articles):6d}")
            
            if len(found) == 0:
                break
            if page % 20 == 0:
                save(f"Checkpoint")
            time.sleep(2)
            
        except Exception as e:
            print(f"    ❌ {e}")
            time.sleep(5)
            break
    
    save(f"After '{keyword}'")

# ══════════════════════════════════════════
# ROUND 4: Scrape Baodautu.vn directly
# ══════════════════════════════════════════
print("\n" + "="*55)
print("ROUND 4: Baodautu.vn (Investment newspaper)")
print("="*55)

for page in range(1, 200):
    url = f"https://baodautu.vn/chung-khoan-d8/trang-{page}.html"
    try:
        resp = requests.get(url, headers=headers, timeout=15)
        resp.encoding = "utf-8"
        soup = BeautifulSoup(resp.text, "html.parser")
        
        found = []
        for a in soup.find_all("a", href=True):
            href  = a.get("href", "")
            title = a.get_text(strip=True)
            if (len(title) > 20
                    and "baodautu.vn" in href
                    and href not in seen_urls
                    and any(x in href for x in ["-d8-", "/chung-khoan", "/tai-chinh"])):
                found.append({
                    "title": title, "url": href,
                    "date": "", "description": "",
                    "source": "Baodautu.vn", "keyword": "chứng khoán"
                })
                seen_urls.add(href)
        
        articles.extend(found)
        print(f"  Page {page:3d} | +{len(found):3d} | Total: {len(articles):6d}")
        
        if len(found) == 0:
            break
        if page % 50 == 0:
            save("Baodautu checkpoint")
        time.sleep(2)
        
    except Exception as e:
        print(f"  ❌ Page {page}: {e}")
        time.sleep(5)

# ══════════════════════════════════════════
# FINAL
# ══════════════════════════════════════════
df = save("FINAL")
print(f"\n{'='*55}")
print(f"✅ COMPLETE: {len(df)} articles")
print(f"📁 {output_path}")
print(f"\nBy source:\n{df['source'].value_counts().to_string()}")

✅ Resuming from 17658 articles

ROUND 2: Extended Google News RSS queries
  [ 1/35] chứng+khoán+2018                              | + 13 | Total:  17671
  [ 2/35] chứng+khoán+2019                              | +  4 | Total:  17675
  [ 3/35] chứng+khoán+2020                              | +  9 | Total:  17684
  [ 4/35] chứng+khoán+2021                              | +  6 | Total:  17690
  [ 5/35] chứng+khoán+2022                              | +  7 | Total:  17697
  [ 6/35] chứng+khoán+2023                              | + 14 | Total:  17711
  [ 7/35] chứng+khoán+2024                              | +  8 | Total:  17719
  [ 8/35] chứng+khoán+2025                              | + 41 | Total:  17760
  [ 9/35] VN-Index+2018                                 | +  7 | Total:  17767
  [10/35] VN-Index+2019                                 | + 12 | Total:  17779
  [11/35] VN-Index+2020                                 | +  2 | Total:  17781
  [12/35] VN-Index+2021                                 |

In [8]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
import json
import re
import time
import random
from concurrent.futures import ThreadPoolExecutor, as_completed
from email.utils import parsedate_to_datetime
from datetime import datetime

INPUT_FILE = "vietnam_articles.csv"
OUTPUT_FILE = "vietnam_articles_dates_filled.csv"

# Start with 5 workers. Increase only if the websites tolerate it.
MAX_WORKERS = 5
TIMEOUT = 20
MAX_RETRIES = 3

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/140.0 Safari/537.36"
    ),
    "Accept-Language": "vi-VN,vi;q=0.9,en;q=0.8",
}

session = requests.Session()
session.headers.update(HEADERS)


def clean_date(value):
    """Convert a date/time string to YYYY-MM-DD."""
    if not value:
        return None

    value = str(value).strip()

    # ISO date/time, e.g. 2025-08-15 or 2025-08-15T09:30:00+07:00
    m = re.search(r"\b(20\d{2}-\d{2}-\d{2})\b", value)
    if m:
        return m.group(1)

    # RFC date, e.g. Fri, 06 Feb 2026 08:00:00 GMT
    try:
        return parsedate_to_datetime(value).date().isoformat()
    except Exception:
        pass

    # Other common formats
    for fmt in [
        "%d/%m/%Y",
        "%d-%m-%Y",
        "%Y/%m/%d",
        "%Y.%m.%d",
    ]:
        try:
            return datetime.strptime(value[:10], fmt).date().isoformat()
        except Exception:
            pass

    return None


def date_from_jsonld(obj):
    """Find datePublished in JSON-LD, including nested structures."""
    if isinstance(obj, dict):
        for key in ["datePublished", "dateCreated"]:
            if obj.get(key):
                d = clean_date(obj[key])
                if d:
                    return d

        # Some pages use @graph
        for value in obj.values():
            d = date_from_jsonld(value)
            if d:
                return d

    elif isinstance(obj, list):
        for item in obj:
            d = date_from_jsonld(item)
            if d:
                return d

    return None


def extract_date(html):
    soup = BeautifulSoup(html, "html.parser")

    # 1. JSON-LD: usually the best source
    for script in soup.find_all("script", type="application/ld+json"):
        try:
            data = json.loads(script.string or script.get_text())
            d = date_from_jsonld(data)
            if d:
                return d, "JSON-LD"
        except Exception:
            continue

    # 2. OpenGraph / article metadata
    meta_names = [
        "article:published_time",
        "article:published",
        "datePublished",
        "date",
        "pubdate",
        "publishdate",
        "publication_date",
        "DC.date",
        "DC.date.issued",
    ]

    for name in meta_names:
        tag = soup.find("meta", attrs={"property": name})
        if not tag:
            tag = soup.find("meta", attrs={"name": name})

        if tag and tag.get("content"):
            d = clean_date(tag["content"])
            if d:
                return d, f"meta:{name}"

    # 3. <time datetime="...">
    for tag in soup.find_all("time"):
        value = tag.get("datetime") or tag.get_text(" ", strip=True)
        d = clean_date(value)
        if d:
            return d, "time"

    # 4. Common HTML attributes/classes
    patterns = [
        r'"datePublished"\s*:\s*"([^"]+)"',
        r'"dateCreated"\s*:\s*"([^"]+)"',
        r'"published_time"\s*:\s*"([^"]+)"',
        r'"publishDate"\s*:\s*"([^"]+)"',
        r'"publish_date"\s*:\s*"([^"]+)"',
    ]

    for pattern in patterns:
        m = re.search(pattern, html, flags=re.I)
        if m:
            d = clean_date(m.group(1))
            if d:
                return d, "HTML/regex"

    return None, None


def get_article_date(row):
    url = str(row["url"]).strip()

    for attempt in range(MAX_RETRIES):
        try:
            response = session.get(
                url,
                timeout=TIMEOUT,
                allow_redirects=True,
            )

            if response.status_code == 200:
                date_value, method = extract_date(response.text)

                return {
                    "index": row["index"],
                    "date": date_value,
                    "method": method,
                    "final_url": response.url,
                    "status": "OK" if date_value else "NO_DATE",
                }

            # Retry temporary/server/rate-limit errors
            if response.status_code in [429, 500, 502, 503, 504]:
                time.sleep((2 ** attempt) + random.random())
                continue

            return {
                "index": row["index"],
                "date": None,
                "method": None,
                "final_url": response.url,
                "status": f"HTTP_{response.status_code}",
            }

        except Exception as e:
            if attempt < MAX_RETRIES - 1:
                time.sleep((2 ** attempt) + random.random())
            else:
                return {
                    "index": row["index"],
                    "date": None,
                    "method": None,
                    "final_url": None,
                    "status": type(e).__name__,
                }


def main():
    df = pd.read_csv(INPUT_FILE)

    # Keep a backup of the original date column.
    df["date_original"] = df["date"]

    # Only process rows where date is missing.
    missing_mask = df["date"].isna() | (
        df["date"].astype(str).str.strip().isin(["", "nan", "None"])
    )

    missing = df.loc[missing_mask].copy()
    missing["index"] = missing.index

    print(f"Total articles: {len(df):,}")
    print(f"Already have dates: {(~missing_mask).sum():,}")
    print(f"Need dates: {missing_mask.sum():,}")
    print("\nFetching dates from article pages...\n")

    results = []

    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        futures = [
            executor.submit(get_article_date, row)
            for _, row in missing.iterrows()
        ]

        for n, future in enumerate(as_completed(futures), start=1):
            result = future.result()
            results.append(result)

            if n % 100 == 0 or n == len(futures):
                found = sum(r["date"] is not None for r in results)
                print(
                    f"Processed {n:,}/{len(futures):,} | "
                    f"dates found: {found:,}"
                )

    # Fill only missing dates. Never overwrite dates you already had.
    result_df = pd.DataFrame(results).set_index("index")

    for idx, result in result_df.iterrows():
        if pd.notna(result["date"]):
            df.loc[idx, "date"] = result["date"]

    # Add audit columns so you can see where recovered dates came from.
    df["date_source"] = ""
    df["date_status"] = ""

    for idx, result in result_df.iterrows():
        if pd.notna(result["date"]):
            df.loc[idx, "date_source"] = result["method"] or ""
            df.loc[idx, "date_status"] = result["status"]
        else:
            df.loc[idx, "date_status"] = result["status"]

    df.to_csv(OUTPUT_FILE, index=False, encoding="utf-8-sig")

    found = sum(pd.notna(result_df["date"]))
    remaining = missing_mask.sum() - found

    print("\nDONE")
    print(f"New dates recovered: {found:,}")
    print(f"Still missing: {remaining:,}")
    print(f"Saved as: {OUTPUT_FILE}")


if __name__ == "__main__":
    main()

Total articles: 18,135
Already have dates: 5,337
Need dates: 12,798

Fetching dates from article pages...

Processed 100/12,798 | dates found: 2
Processed 200/12,798 | dates found: 14
Processed 300/12,798 | dates found: 14
Processed 400/12,798 | dates found: 14
Processed 500/12,798 | dates found: 14
Processed 600/12,798 | dates found: 15
Processed 700/12,798 | dates found: 16
Processed 800/12,798 | dates found: 17
Processed 900/12,798 | dates found: 20
Processed 1,000/12,798 | dates found: 20
Processed 1,100/12,798 | dates found: 23
Processed 1,200/12,798 | dates found: 24
Processed 1,300/12,798 | dates found: 29
Processed 1,400/12,798 | dates found: 31
Processed 1,500/12,798 | dates found: 33
Processed 1,600/12,798 | dates found: 34
Processed 1,700/12,798 | dates found: 35
Processed 1,800/12,798 | dates found: 35
Processed 1,900/12,798 | dates found: 35
Processed 2,000/12,798 | dates found: 35
Processed 2,100/12,798 | dates found: 38
Processed 2,200/12,798 | dates found: 38
Processed 

In [9]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
import json
import re
import time
import random
from urllib.parse import urlparse
from concurrent.futures import ThreadPoolExecutor, as_completed
from email.utils import parsedate_to_datetime
from datetime import datetime

# ============================================================
# VERSION 2 — Vietnamese Financial News Date Recovery
# Designed for: VnExpress, CafeF, Vietstock and similar sites
# ============================================================

INPUT_FILE = "vietnam_articles_dates_filled.csv"
OUTPUT_FILE = "vietnam_articles_dates_v2.csv"
UNRESOLVED_FILE = "vietnam_articles_unresolved.csv"

MAX_WORKERS = 5
TIMEOUT = 25
MAX_RETRIES = 3
SAVE_EVERY = 250

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/140.0 Safari/537.36"
    ),
    "Accept-Language": "vi-VN,vi;q=0.9,en-US;q=0.8,en;q=0.7",
}

session = requests.Session()
session.headers.update(HEADERS)

# Vietnamese month names
VI_MONTHS = {
    "tháng một": 1, "tháng 1": 1,
    "tháng hai": 2, "tháng 2": 2,
    "tháng ba": 3, "tháng 3": 3,
    "tháng tư": 4, "tháng 4": 4,
    "tháng năm": 5, "tháng 5": 5,
    "tháng sáu": 6, "tháng 6": 6,
    "tháng bảy": 7, "tháng 7": 7,
    "tháng tám": 8, "tháng 8": 8,
    "tháng chín": 9, "tháng 9": 9,
    "tháng mười": 10, "tháng 10": 10,
    "tháng mười một": 11, "tháng 11": 11,
    "tháng mười hai": 12, "tháng 12": 12,
}

DATE_REGEXES = [
    # dd/mm/yyyy, dd-mm-yyyy, dd.mm.yyyy
    r"\b([0-3]?\d)[/\-.]([0-1]?\d)[/\-.](20\d{2})\b",

    # yyyy-mm-dd
    r"\b(20\d{2})[-/](0?[1-9]|1[0-2])[-/](0?[1-9]|[12]\d|3[01])\b",

    # dd tháng mm yyyy / dd tháng <Vietnamese month> yyyy
    r"\b([0-3]?\d)\s+tháng\s+([0-1]?\d)\s*,?\s*(20\d{2})\b",
    r"\b([0-3]?\d)\s+tháng\s+(một|hai|ba|tư|năm|sáu|bảy|tám|chín|mười(?:\s+một|\s+hai)?)\s*,?\s*(20\d{2})\b",

    # dd Month yyyy in English
    r"\b([0-3]?\d)\s+(January|February|March|April|May|June|July|August|September|October|November|December)\s+(20\d{2})\b",
]


def clean_date(value):
    if value is None:
        return None

    value = str(value).strip()

    # ISO / timestamp
    m = re.search(r"\b(20\d{2})-(\d{1,2})-(\d{1,2})\b", value)
    if m:
        return make_date(m.group(1), m.group(2), m.group(3))

    # RFC
    try:
        return parsedate_to_datetime(value).date().isoformat()
    except Exception:
        pass

    # Common formats
    for fmt in [
        "%d/%m/%Y",
        "%d-%m-%Y",
        "%d.%m.%Y",
        "%Y/%m/%d",
        "%Y.%m.%d",
    ]:
        try:
            return datetime.strptime(value[:10], fmt).date().isoformat()
        except Exception:
            pass

    return None


def make_date(y, m, d):
    try:
        dt = datetime(int(y), int(m), int(d))
        if 2010 <= dt.year <= 2026:
            return dt.date().isoformat()
    except Exception:
        return None
    return None


def date_from_jsonld(obj):
    """Find datePublished only. Do NOT use dateModified as publication date."""
    if isinstance(obj, dict):
        for key in ["datePublished", "dateCreated"]:
            if obj.get(key):
                d = clean_date(obj[key])
                if d:
                    return d, "JSON-LD datePublished"

        for value in obj.values():
            result = date_from_jsonld(value)
            if result:
                return result

    elif isinstance(obj, list):
        for item in obj:
            result = date_from_jsonld(item)
            if result:
                return result

    return None


def extract_microdata(soup):
    # Schema.org microdata
    tags = soup.find_all(attrs={"itemprop": "datePublished"})
    for tag in tags:
        value = tag.get("content") or tag.get("datetime") or tag.get_text(" ", strip=True)
        d = clean_date(value)
        if d:
            return d, "microdata datePublished"

    return None


def extract_meta_date(soup):
    names = [
        "article:published_time",
        "article:published",
        "datePublished",
        "datepublished",
        "pubdate",
        "publishdate",
        "publication_date",
        "DC.date",
        "DC.date.issued",
        "date",
    ]

    for name in names:
        tag = soup.find("meta", attrs={"property": name})
        if not tag:
            tag = soup.find("meta", attrs={"name": name})

        if tag and tag.get("content"):
            d = clean_date(tag["content"])
            if d:
                return d, f"meta {name}"

    return None


def extract_visible_date(soup):
    """
    Search text around publication-date indicators.
    Avoid blindly accepting every date in an article because financial
    articles contain many historical dates.
    """

    # First inspect likely date containers.
    selectors = [
        "time",
        "[class*='date']",
        "[class*='time']",
        "[class*='publish']",
        "[class*='author']",
        "[class*='meta']",
        "[class*='detail']",
        "[class*='article']",
    ]

    candidate_texts = []

    for selector in selectors:
        try:
            for tag in soup.select(selector):
                text = tag.get_text(" ", strip=True)
                if text:
                    candidate_texts.append(text)
        except Exception:
            pass

    # Also inspect the beginning of the page text.
    body_text = soup.get_text(" ", strip=True)
    candidate_texts.append(body_text[:12000])

    indicators = [
        "ngày đăng",
        "đăng ngày",
        "đăng lúc",
        "đăng tải",
        "cập nhật",
        "xuất bản",
        "published",
        "posted",
    ]

    # Strong pattern: indicator close to date
    for text in candidate_texts:
        lower = text.lower()

        if not any(x in lower for x in indicators):
            continue

        for pattern in DATE_REGEXES:
            m = re.search(pattern, text, flags=re.I)
            if not m:
                continue

            groups = m.groups()

            try:
                # yyyy-mm-dd pattern
                if len(groups[0]) == 4:
                    d = make_date(groups[0], groups[1], groups[2])
                # English month
                elif groups[1].lower() in [
                    "january", "february", "march", "april", "may",
                    "june", "july", "august", "september", "october",
                    "november", "december"
                ]:
                    month = datetime.strptime(groups[1], "%B").month
                    d = make_date(groups[2], month, groups[0])
                # Vietnamese month word
                elif not groups[1].isdigit():
                    month = {
                        "một": 1, "hai": 2, "ba": 3, "tư": 4,
                        "năm": 5, "sáu": 6, "bảy": 7, "tám": 8,
                        "chín": 9, "mười": 10,
                        "mười một": 11, "mười hai": 12
                    }.get(groups[1].lower())
                    d = make_date(groups[2], month, groups[0]) if month else None
                else:
                    d = make_date(groups[2], groups[1], groups[0])

                if d:
                    return d, "visible date near publication indicator"

            except Exception:
                continue

    # Search explicit Vietnamese format throughout page text,
    # but only if it appears near the top of the article.
    top_text = body_text[:18000]

    for pattern in DATE_REGEXES:
        m = re.search(pattern, top_text, flags=re.I)
        if m:
            groups = m.groups()

            try:
                if len(groups[0]) == 4:
                    d = make_date(groups[0], groups[1], groups[2])
                elif groups[1].lower() in [
                    "january", "february", "march", "april", "may",
                    "june", "july", "august", "september", "october",
                    "november", "december"
                ]:
                    month = datetime.strptime(groups[1], "%B").month
                    d = make_date(groups[2], month, groups[0])
                elif not groups[1].isdigit():
                    month = {
                        "một": 1, "hai": 2, "ba": 3, "tư": 4,
                        "năm": 5, "sáu": 6, "bảy": 7, "tám": 8,
                        "chín": 9, "mười": 10,
                        "mười một": 11, "mười hai": 12
                    }.get(groups[1].lower())
                    d = make_date(groups[2], month, groups[0]) if month else None
                else:
                    d = make_date(groups[2], groups[1], groups[0])

                if d:
                    return d, "visible date near top of page"
            except Exception:
                pass

    return None


def extract_date_from_url(url):
    """
    Some publishers put the publication date in the URL.
    Only accept plausible 2018-2025 dates.
    """
    patterns = [
        r"(20(?:18|19|20|21|22|23|24|25))[-_/](\d{1,2})[-_/](\d{1,2})",
        r"[-_/](\d{1,2})[-_/](\d{1,2})[-_/](20(?:18|19|20|21|22|23|24|25))",
    ]

    for pattern in patterns:
        m = re.search(pattern, url)
        if m:
            g = m.groups()

            if len(g[0]) == 4:
                d = make_date(g[0], g[1], g[2])
            else:
                d = make_date(g[2], g[0], g[1])

            if d:
                return d, "URL date pattern"

    return None


def publisher_hint(url):
    host = urlparse(url).netloc.lower()

    if "vnexpress.net" in host:
        return "VnExpress"
    if "cafef.vn" in host:
        return "CafeF"
    if "vietstock.vn" in host:
        return "Vietstock"
    if "vneconomy.vn" in host:
        return "VnEconomy"
    if "dantri.com.vn" in host:
        return "Dantri"
    if "tuoitre.vn" in host:
        return "TuoiTre"
    if "thanhnien.vn" in host:
        return "ThanhNien"
    if "nhandan.vn" in host:
        return "NhanDan"

    return "Other"


def extract_date(html, url):
    soup = BeautifulSoup(html, "html.parser")

    # Method 1 — JSON-LD
    for script in soup.find_all("script", type="application/ld+json"):
        try:
            raw = script.string or script.get_text()
            data = json.loads(raw)
            result = date_from_jsonld(data)
            if result:
                return result
        except Exception:
            continue

    # Method 2 — schema.org microdata
    result = extract_microdata(soup)
    if result:
        return result

    # Method 3 — meta tags
    result = extract_meta_date(soup)
    if result:
        return result

    # Method 4 — visible Vietnamese/English publication date
    result = extract_visible_date(soup)
    if result:
        return result

    # Method 5 — URL pattern
    result = extract_date_from_url(url)
    if result:
        return result

    return None, None


def fetch_one(index, url):
    if not url or str(url).lower() in ["nan", "none"]:
        return {
            "index": index,
            "date": None,
            "method": None,
            "publisher": None,
            "status": "NO_URL",
        }

    url = str(url).strip()
    publisher = publisher_hint(url)

    for attempt in range(MAX_RETRIES):
        try:
            response = session.get(
                url,
                timeout=TIMEOUT,
                allow_redirects=True,
            )

            if response.status_code == 200:
                date, method = extract_date(response.text, response.url)

                return {
                    "index": index,
                    "date": date,
                    "method": method,
                    "publisher": publisher,
                    "status": "FOUND" if date else "NO_DATE",
                }

            if response.status_code in [429, 500, 502, 503, 504]:
                time.sleep((2 ** attempt) + random.uniform(0.5, 1.5))
                continue

            return {
                "index": index,
                "date": None,
                "method": None,
                "publisher": publisher,
                "status": f"HTTP_{response.status_code}",
            }

        except requests.RequestException as e:
            if attempt < MAX_RETRIES - 1:
                time.sleep((2 ** attempt) + random.uniform(0.5, 1.5))
            else:
                return {
                    "index": index,
                    "date": None,
                    "method": None,
                    "publisher": publisher,
                    "status": type(e).__name__,
                }

    return {
        "index": index,
        "date": None,
        "method": None,
        "publisher": publisher,
        "status": "FAILED_AFTER_RETRIES",
    }


def is_missing(value):
    return pd.isna(value) or str(value).strip().lower() in [
        "", "nan", "none", "nat"
    ]


def main():

    print("=" * 65)
    print("VERSION 2 — VIETNAMESE FINANCIAL NEWS DATE RECOVERY")
    print("=" * 65)

    df = pd.read_csv(INPUT_FILE)

    # Make sure date column exists
    if "date" not in df.columns:
        raise ValueError("CSV does not contain a 'date' column.")

    if "url" not in df.columns:
        raise ValueError("CSV does not contain a 'url' column.")

    # Preserve original dates
    if "date_original" not in df.columns:
        df["date_original"] = df["date"]

    # Preserve previous recovery information if present
    if "date_source" not in df.columns:
        df["date_source"] = ""

    if "date_status" not in df.columns:
        df["date_status"] = ""

    missing_mask = df["date"].apply(is_missing)
    missing = df.loc[missing_mask].copy()

    print(f"Total articles: {len(df):,}")
    print(f"Dates already available: {(~missing_mask).sum():,}")
    print(f"Dates still missing: {missing_mask.sum():,}")
    print()

    if len(missing) == 0:
        print("No missing dates. Nothing to process.")
        return

    results = []

    tasks = [
        (idx, row["url"])
        for idx, row in missing.iterrows()
    ]

    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:

        futures = {
            executor.submit(fetch_one, idx, url): idx
            for idx, url in tasks
        }

        for count, future in enumerate(as_completed(futures), start=1):

            result = future.result()
            results.append(result)

            if count % 100 == 0 or count == len(futures):
                found = sum(
                    1 for r in results
                    if r["date"] is not None
                )

                print(
                    f"Processed {count:,}/{len(futures):,} | "
                    f"Recovered this run: {found:,}"
                )

            # Periodic checkpoint
            if count % SAVE_EVERY == 0:
                pd.DataFrame(results).to_csv(
                    "v2_recovery_checkpoint.csv",
                    index=False,
                    encoding="utf-8-sig"
                )

    result_df = pd.DataFrame(results).set_index("index")

    # Fill ONLY missing dates.
    # Existing dates are never overwritten.
    for idx, result in result_df.iterrows():

        if result["date"] is not None:
            df.loc[idx, "date"] = result["date"]
            df.loc[idx, "date_source"] = result["method"] or ""
            df.loc[idx, "date_status"] = "V2_RECOVERED"
        else:
            df.loc[idx, "date_status"] = result["status"]

    # Save full dataset
    df.to_csv(
        OUTPUT_FILE,
        index=False,
        encoding="utf-8-sig"
    )

    # Save unresolved rows separately
    unresolved_mask = df["date"].apply(is_missing)

    unresolved = df.loc[unresolved_mask].copy()

    # Add the last attempted result if available
    if len(result_df):
        for idx, result in result_df.iterrows():
            if idx in unresolved.index:
                unresolved.loc[idx, "recovery_method_attempted"] = (
                    result["method"] or ""
                )
                unresolved.loc[idx, "recovery_status"] = result["status"]
                unresolved.loc[idx, "publisher"] = result["publisher"] or ""

    unresolved.to_csv(
        UNRESOLVED_FILE,
        index=False,
        encoding="utf-8-sig"
    )

    recovered = len(missing) - len(unresolved)

    print()
    print("=" * 65)
    print("COMPLETE")
    print("=" * 65)
    print(f"New dates recovered: {recovered:,}")
    print(f"Still unresolved: {len(unresolved):,}")
    print()
    print(f"Full dataset: {OUTPUT_FILE}")
    print(f"Unresolved only: {UNRESOLVED_FILE}")
    print()
    print("Recovery methods used:")
    print(result_df["method"].value_counts(dropna=False).to_string())


if __name__ == "__main__":
    main()

VERSION 2 — VIETNAMESE FINANCIAL NEWS DATE RECOVERY
Total articles: 18,135
Dates already available: 12,055
Dates still missing: 6,080

Processed 100/6,080 | Recovered this run: 100
Processed 200/6,080 | Recovered this run: 200
Processed 300/6,080 | Recovered this run: 300
Processed 400/6,080 | Recovered this run: 400
Processed 500/6,080 | Recovered this run: 500
Processed 600/6,080 | Recovered this run: 600
Processed 700/6,080 | Recovered this run: 700
Processed 800/6,080 | Recovered this run: 800
Processed 900/6,080 | Recovered this run: 898
Processed 1,000/6,080 | Recovered this run: 998
Processed 1,100/6,080 | Recovered this run: 1,098
Processed 1,200/6,080 | Recovered this run: 1,197
Processed 1,300/6,080 | Recovered this run: 1,294
Processed 1,400/6,080 | Recovered this run: 1,393
Processed 1,500/6,080 | Recovered this run: 1,493
Processed 1,600/6,080 | Recovered this run: 1,593
Processed 1,700/6,080 | Recovered this run: 1,693
Processed 1,800/6,080 | Recovered this run: 1,793
Pro

In [11]:
import pandas as pd
import requests, re, json, time, random
from bs4 import BeautifulSoup
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime

INPUT_FILE = "vietnam_articles_dates_filled.csv"
OUTPUT_FILE = "vietnam_articles_dates_v3.csv"
UNRESOLVED_FILE = "vietnam_articles_unresolved_v3.csv"
REVIEW_FILE = "vietnam_articles_review_v3.csv"

MAX_WORKERS = 3
TIMEOUT = 30
RETRIES = 3

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 Chrome/140.0 Safari/537.36",
    "Accept-Language": "vi-VN,vi;q=0.9,en;q=0.8"
}

session = requests.Session()
session.headers.update(HEADERS)

def valid(y,m,d):
    try:
        y,m,d = int(y),int(m),int(d)
        if not 2018 <= y <= 2025: return None
        return datetime(y,m,d).strftime("%Y-%m-%d")
    except: return None

def parse_date(v):
    if v is None: return None
    v = str(v).strip()
    patterns = [
        (r"\b(20\d{2})-(\d{1,2})-(\d{1,2})\b", (1,2,3)),
        (r"\b(\d{1,2})/(\d{1,2})/(20\d{2})\b", (3,2,1)),
        (r"\b(\d{1,2})-(\d{1,2})-(20\d{2})\b", (3,2,1)),
        (r"\b(\d{1,2})\.(\d{1,2})\.(20\d{2})\b", (3,2,1)),
    ]
    for p, ix in patterns:
        m = re.search(p,v)
        if m: 
            d=valid(*(m.group(i) for i in ix))
            if d: return d
    return None

def jsonld_date(obj):
    if isinstance(obj,dict):
        for k in ("datePublished","dateCreated"):
            if k in obj:
                d=parse_date(obj[k])
                if d: return d
        for x in obj.values():
            d=jsonld_date(x)
            if d: return d
    elif isinstance(obj,list):
        for x in obj:
            d=jsonld_date(x)
            if d: return d
    return None

def get_jsonld(soup):
    for s in soup.find_all("script",type=re.compile("application/ld\\+json",re.I)):
        try:
            d=json.loads(s.string or s.get_text())
            x=jsonld_date(d)
            if x: return x,"JSON-LD datePublished"
        except: pass
    return None,None

def get_meta(soup):
    for prop in ("article:published_time","article:published"):
        t=soup.find("meta",attrs={"property":prop})
        if t:
            d=parse_date(t.get("content"))
            if d: return d,"meta "+prop
    for t in soup.find_all("meta",attrs={"itemprop":"datePublished"}):
        d=parse_date(t.get("content"))
        if d: return d,"schema datePublished"
    return None,None

def get_time(soup):
    candidates=[]
    words=("đăng","xuất bản","published","posted","publication")
    for t in soup.find_all("time"):
        value=t.get("datetime") or t.get("content") or t.get_text(" ",strip=True)
        context=(t.get_text(" ",strip=True)+" "+" ".join(t.get("class",[])))
        if any(w in context.lower() for w in words):
            d=parse_date(value)
            if d: candidates.append(d)
    u=list(dict.fromkeys(candidates))
    return (u[0],"time publication element") if len(u)==1 else (None,None)

def get_publisher(soup,url):
    host=url.lower()
    selectors=[]
    name=""
    if "vnexpress.net" in host:
        name="VnExpress"
        selectors=[".date",".date-time",".article-header .date",".header-content .date",".time"]
    elif "cafef.vn" in host:
        name="CafeF"
        selectors=[".time",".date",".newsdetail-time",".detail-time"]
    elif "vietstock.vn" in host:
        name="Vietstock"
        selectors=[".date",".time",".article-date",".date-time",".news-date"]
    else:
        return None,None
    dates=[]
    for sel in selectors:
        for t in soup.select(sel):
            d=parse_date(t.get_text(" ",strip=True))
            if d: dates.append(d)
    u=list(dict.fromkeys(dates))
    return (u[0],name+" publication element") if len(u)==1 else (None,None)

def get_label_date(soup):
    labels=("ngày đăng","đăng ngày","đăng lúc","đăng tải","xuất bản","published","posted")
    dates=[]
    for t in soup.find_all(["div","span","p","li","time"]):
        text=t.get_text(" ",strip=True)
        if not text or len(text)>500: continue
        low=text.lower()
        if any(x in low for x in labels):
            d=parse_date(text)
            if d: dates.append(d)
    u=list(dict.fromkeys(dates))
    return (u[0],"publication label") if len(u)==1 else (None,None)

def extract(html,url):
    soup=BeautifulSoup(html,"html.parser")
    for fn in (get_jsonld,get_meta,get_time):
        d,m=fn(soup)
        if d: return d,m
    d,m=get_publisher(soup,url)
    if d: return d,m
    return get_label_date(soup)

def fetch(idx,url):
    if pd.isna(url): return idx,None,None,"NO_URL"
    for attempt in range(RETRIES):
        try:
            r=session.get(str(url).strip(),timeout=TIMEOUT,allow_redirects=True)
            if r.status_code==200:
                d,m=extract(r.text,r.url)
                return idx,d,m,"FOUND" if d else "NO_STRONG_DATE"
            if r.status_code in (429,500,502,503,504):
                time.sleep(2**attempt+random.random()); continue
            return idx,None,None,f"HTTP_{r.status_code}"
        except Exception as e:
            if attempt<RETRIES-1:
                time.sleep(2**attempt+random.random())
            else:
                return idx,None,None,type(e).__name__
    return idx,None,None,"FAILED"

def main():
    df=pd.read_csv(INPUT_FILE)
    if "url" not in df.columns or "date" not in df.columns:
        raise ValueError("CSV must contain 'url' and 'date' columns.")

    if "date_original" not in df: df["date_original"]=df["date"]
    if "date_recovery_method" not in df: df["date_recovery_method"]=""
    if "date_recovery_status" not in df: df["date_recovery_status"]=""

    missing=df["date"].isna() | df["date"].astype(str).str.strip().isin(["","nan","None"])
    rows=[(i,row["url"]) for i,row in df.loc[missing].iterrows()]

    print(f"Total: {len(df):,}")
    print(f"Already dated: {(~missing).sum():,}")
    print(f"To check: {missing.sum():,}")
    print("Existing dates will NOT be overwritten.")

    results=[]
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
        jobs=[ex.submit(fetch,i,u) for i,u in rows]
        for n,f in enumerate(as_completed(jobs),1):
            results.append(f.result())
            if n%100==0 or n==len(jobs):
                found=sum(x[1] is not None for x in results)
                print(f"Checked {n:,}/{len(jobs):,} | recovered {found:,}")

    for idx,d,m,status in results:
        empty=pd.isna(df.at[idx,"date"]) or str(df.at[idx,"date"]).strip() in ("","nan","None")
        if empty and d:
            df.at[idx,"date"]=d
            df.at[idx,"date_recovery_method"]=m
            df.at[idx,"date_recovery_status"]="RECOVERED"
        elif empty:
            df.at[idx,"date_recovery_status"]=status

    df.to_csv(OUTPUT_FILE,index=False,encoding="utf-8-sig")
    unresolved=df[df["date"].isna() | df["date"].astype(str).str.strip().isin(["","nan","None"])]
    unresolved.to_csv(UNRESOLVED_FILE,index=False,encoding="utf-8-sig")

    review=df[df["date_recovery_method"].isin([
        "publication label","time publication element",
        "VnExpress publication element","CafeF publication element",
        "Vietstock publication element"
    ])]
    review.to_csv(REVIEW_FILE,index=False,encoding="utf-8-sig")

    print("\nDONE")
    print(f"Final dated: {(~(df['date'].isna())).sum():,}")
    print(f"Still unresolved: {len(unresolved):,}")
    print(f"Full: {OUTPUT_FILE}")
    print(f"Unresolved: {UNRESOLVED_FILE}")
    print(f"Review: {REVIEW_FILE}")

if __name__=="__main__":
    main()

Total: 18,135
Already dated: 12,055
To check: 6,080
Existing dates will NOT be overwritten.
Checked 100/6,080 | recovered 26
Checked 200/6,080 | recovered 90
Checked 300/6,080 | recovered 166
Checked 400/6,080 | recovered 255
Checked 500/6,080 | recovered 321
Checked 600/6,080 | recovered 375
Checked 700/6,080 | recovered 408
Checked 800/6,080 | recovered 440
Checked 900/6,080 | recovered 470
Checked 1,000/6,080 | recovered 537
Checked 1,100/6,080 | recovered 622
Checked 1,200/6,080 | recovered 714
Checked 1,300/6,080 | recovered 739
Checked 1,400/6,080 | recovered 799
Checked 1,500/6,080 | recovered 894
Checked 1,600/6,080 | recovered 989
Checked 1,700/6,080 | recovered 1,079
Checked 1,800/6,080 | recovered 1,168
Checked 1,900/6,080 | recovered 1,256
Checked 2,000/6,080 | recovered 1,350
Checked 2,100/6,080 | recovered 1,442
Checked 2,200/6,080 | recovered 1,534
Checked 2,300/6,080 | recovered 1,617
Checked 2,400/6,080 | recovered 1,694
Checked 2,500/6,080 | recovered 1,720
Checked 2,

In [13]:
import pandas as pd
import re
import os
from datetime import datetime, date

input_path  = os.path.expanduser("~/Desktop/vietnam_articles_dates_v3.csv")
output_path = os.path.expanduser("~/Desktop/vietnam_articles_final.csv")

df = pd.read_csv(input_path, encoding="utf-8-sig")
print(f"✅ Loaded: {len(df)} articles")

# ══════════════════════════════════════════
# STEP 1: Parse dates
# ══════════════════════════════════════════
def parse_date(date_str):
    if pd.isna(date_str) or str(date_str).strip() == "":
        return None
    s = str(date_str).strip()
    s = re.sub(r'\s*(GMT|UTC|ICT|\+\d{2}:?\d{2}|-\d{2}:?\d{2})\s*$', '', s).strip()
    s = re.sub(r'^Thứ\s+\w+,?\s*', '', s).strip()
    formats = [
        "%a, %d %b %Y %H:%M:%S",
        "%a, %d %b %Y",
        "%Y-%m-%dT%H:%M:%S",
        "%Y-%m-%dT%H:%M:%S.%f",
        "%Y-%m-%d %H:%M:%S",
        "%Y-%m-%d",
        "%d/%m/%Y %H:%M",
        "%d/%m/%Y",
        "%d-%m-%Y",
        "%d.%m.%Y",
        "%B %d, %Y",
        "%b %d, %Y",
        "%d %b %Y",
    ]
    for fmt in formats:
        try:
            return datetime.strptime(s, fmt).date()
        except:
            continue
    m = re.search(r'(\d{4})-(\d{2})-(\d{2})', s)
    if m:
        try:
            return date(int(m.group(1)), int(m.group(2)), int(m.group(3)))
        except:
            pass
    m = re.search(r'(\d{1,2})/(\d{1,2})/(20\d{2})', s)
    if m:
        try:
            return date(int(m.group(3)), int(m.group(2)), int(m.group(1)))
        except:
            pass
    return None

print("⏳ Parsing dates...")
df["date_parsed"] = df["date"].apply(parse_date)
print(f"  ✅ Parsed:  {df['date_parsed'].notna().sum()}")
print(f"  ❌ Failed:  {df['date_parsed'].isna().sum()}")

failed = df[df["date_parsed"].isna()]["date"].dropna().head(10)
if len(failed) > 0:
    print(f"\n  Sample failed dates:")
    print(failed.to_string())

# ══════════════════════════════════════════
# STEP 2: Filter 2018–2026
# ══════════════════════════════════════════
START = date(2018, 1, 1)
END   = date(2026, 12, 31)

df_clean = df[df["date_parsed"].notna()].copy()
df_clean = df_clean[
    (df_clean["date_parsed"] >= START) &
    (df_clean["date_parsed"] <= END)
].copy()

print(f"\n📅 Within 2018–2026: {len(df_clean)}")
df_clean["year"] = pd.to_datetime(df_clean["date_parsed"]).dt.year
print(f"\nBy year:\n{df_clean['year'].value_counts().sort_index().to_string()}")

# ══════════════════════════════════════════
# STEP 3: Remove duplicates
# ══════════════════════════════════════════
before = len(df_clean)
df_clean = df_clean.drop_duplicates(subset=["url"])
print(f"\n🔁 Removed {before - len(df_clean)} duplicate URLs")
before = len(df_clean)
df_clean = df_clean.drop_duplicates(subset=["title"])
print(f"🔁 Removed {before - len(df_clean)} duplicate titles")

# ══════════════════════════════════════════
# STEP 4: Filter finance-related only
# ══════════════════════════════════════════
finance_keywords = [
    "chứng khoán", "cổ phiếu", "vn-index", "vnindex", "hose", "hnx",
    "thị trường", "nhà đầu tư", "tăng điểm", "giảm điểm", "thanh khoản",
    "cổ phần", "niêm yết", "phái sinh", "etf", "vn30", "upcom",
    "ngân hàng", "lãi suất", "tỷ giá", "lạm phát", "kinh tế",
    "doanh nghiệp", "tài chính", "đầu tư", "ipo", "thoái vốn",
    "cổ tức", "margin", "room ngoại", "ngoại khối", "trái phiếu",
    "vietcombank", "vietinbank", "bidv", "vpbank", "techcombank",
    "vingroup", "vinhomes", "masan", "hoa phat", "vinamilk", "fpt",
    "tăng mạnh", "giảm mạnh", "phiên giao dịch", "khối ngoại",
    "chứng chỉ quỹ", "hợp đồng tương lai", "quyền chọn",
    "stock", "share", "equity", "market index", "investor",
    "securities", "investment fund", "volatility", "dividend",
    "vietnam stock", "exchange rate", "interest rate",
]

def is_finance(title, desc=""):
    text = (str(title) + " " + str(desc)).lower()
    return any(kw.lower() in text for kw in finance_keywords)

df_clean["is_finance"] = df_clean.apply(
    lambda r: is_finance(r["title"], r.get("description", "")), axis=1)

before = len(df_clean)
df_finance = df_clean[df_clean["is_finance"]].copy()
print(f"\n💰 Finance articles kept: {len(df_finance)}")
print(f"🗑️  Non-finance removed:   {before - len(df_finance)}")

# ══════════════════════════════════════════
# STEP 5: Clean text
# ══════════════════════════════════════════
def clean_text(t):
    if pd.isna(t): return ""
    t = re.sub(r'<[^>]+>', '', str(t))
    t = re.sub(r'http\S+', '', t)
    return re.sub(r'\s+', ' ', t).strip()

df_finance["title"]       = df_finance["title"].apply(clean_text)
df_finance["description"] = df_finance["description"].apply(clean_text)

# ══════════════════════════════════════════
# STEP 6: Format date as DD-MM-YYYY & save
# ══════════════════════════════════════════
df_finance = df_finance.sort_values("date_parsed").reset_index(drop=True)

# Format date as DD-MM-YYYY  ← your requested format
df_finance["date_formatted"] = df_finance["date_parsed"].apply(
    lambda d: d.strftime("%d-%m-%Y") if d else "")

df_final = df_finance[[
    "title", "description", "url", "date_formatted", "source", "keyword"
]].rename(columns={"date_formatted": "date"})

df_final.to_csv(output_path, index=False, encoding="utf-8-sig")

print(f"\n{'='*50}")
print(f"✅ FINAL CLEAN DATASET")
print(f"   Total articles:  {len(df_final)}")
print(f"   Date range:      {df_final['date'].iloc[0]} → {df_final['date'].iloc[-1]}")
print(f"   Saved to:        {output_path}")
print(f"\nBy year:")
print(pd.to_datetime(df_final["date"], format='%d-%m-%Y').dt.year
      .value_counts().sort_index().to_string())
print(f"\nBy source:")
print(df_final["source"].value_counts().to_string())
print(f"\nSample (title + date):")
print(df_final[["title","date"]].head(15).to_string())

✅ Loaded: 18135 articles
⏳ Parsing dates...
  ✅ Parsed:  15809
  ❌ Failed:  2326

📅 Within 2018–2026: 15415

By year:
year
2018     217
2019     270
2020     603
2021     976
2022    1304
2023    1591
2024    2086
2025    3087
2026    5281

🔁 Removed 0 duplicate URLs
🔁 Removed 324 duplicate titles

💰 Finance articles kept: 12995
🗑️  Non-finance removed:   2096

✅ FINAL CLEAN DATASET
   Total articles:  12995
   Date range:      02-01-2018 → 23-09-2026
   Saved to:        /Users/alicenguyen/Desktop/vietnam_articles_final.csv

By year:
date
2018     193
2019     239
2020     529
2021     850
2022    1150
2023    1321
2024    1767
2025    2501
2026    4445

By source:
source
VnExpress                                      8075
CafeF                                           549
Tin nhanh chứng khoán                           533
Vietnam.vn                                      313
Vietstock                                       266
VnEconomy                                       223
Baodaut